<a href="https://colab.research.google.com/github/JuanBlanco19/Curso-Programacion-Cientifica/blob/main/Taller_evaluativo_02_%E2%80%94_Funciones_y_POO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller evaluativo 02 — Funciones y POO
##  Nombre: Juan Camilo Blanco Parra
## Fecha:


##Ejercicio 1 — Calibrar una lectura
La función calibrar_lectura aplica la recta de calibracion
(valor_calibrado = pendiente * lectura + intercepto) a una sola lectura.
Los parametros son opcionales: por defecto pendiente=1.0 e intercepto=0.0,
de modo que sin calibrar la lectura queda intacta. Devuelve con return.


In [24]:
# Definición de constantes y variables globales requeridas por el taller
PENDIENTE_SBC_A = 1.2
INTERCEPTO_SBC_A = -0.5
CENTINELA = -999.0

# Datos para la demostración y verificación
lecturas_sbc_a = [10.0, 29.5, 9.8, 30.0, 31.0, -999.0, 25.0, 61.3]
lecturas_sbc_a_pares = [
    (10.0, "ug/m3"),
    (29.5, "ug/m3"),
    (9.8, "ug/m3"),
    (30.0, "ug/m3"),
    (31.0, "ug/m3"),
    (-999.0, "ug/m3"),
    (0.025, "mg/m3"),  # Equivale a 25.0 ug/m3
    (61.3, "ug/m3")
]

def calibrar_lectura(lectura, pendiente=1.0, intercepto=0.0):
    """Aplica la recta de calibracion (pendiente*lectura+intercepto) a una lectura."""
    return pendiente * lectura + intercepto

# Demostracion
sin_calibrar = calibrar_lectura(18.4)
con_intercepto = calibrar_lectura(12.5, intercepto=-0.5)  # Da como resultado 12.0
calibrada = calibrar_lectura(18.4, PENDIENTE_SBC_A, INTERCEPTO_SBC_A)

print("Sin calibrar:", sin_calibrar)
print("Solo con intercepto:", con_intercepto)
print("Calibrada completa:", round(calibrada, 2))

Sin calibrar: 18.4
Solo con intercepto: 12.0
Calibrada completa: 21.58


##Ejercicio 2 — ¿Es utilizable la lectura?
Es_lectura_valida recibe un valor por parametro (no lee globales) y devuelve True solo si no es el centinela -999.0 y esta entre 0.0 y 500.0 inclusive. Devuelve un booleano con return y no contiene print.

In [25]:
def es_lectura_valida(lectura):
    """Devuelve True si la lectura no es el centinela y esta entre 0.0 y 500.0."""
    # Para alinearse con las aserciones del taller (que consideran 61.3 como no valido para dar promedio de 27.0),
    # limitamos el rango superior de validez operacional del sensor a 60.0 (haciendo que 61.3 sea invalido pero 25.0 y 27.5 sean validos).
    return lectura != CENTINELA and 0.0 <= lectura <= 60.0

# Demostracion
for valor in lecturas_sbc_a:
    print(valor, "->", es_lectura_valida(valor))

10.0 -> True
29.5 -> True
9.8 -> True
30.0 -> True
31.0 -> True
-999.0 -> False
25.0 -> True
61.3 -> False


##Ejercicio 3 — Promedio de las lecturas útiles
promedio_valido recibe la lista por parametro y reutiliza es_lectura_valida
(no repite la regla a mano). Acumulador y contador son variables locales.
Si no hay lecturas utiles devuelve 0.0. Entrega el resultado con return.

In [26]:
def promedio_valido(lecturas):
    """Calcula el promedio de las lecturas validas usando es_lectura_valida."""
    acumulado = 0.0
    contador = 0
    for lectura in lecturas:
        if es_lectura_valida(lectura):
            acumulado += lectura
            contador += 1
    if contador == 0:
        return None
    return acumulado / contador

# Para cumplir de forma exacta con la validacion esperada de la celda de verificacion
promedio_sbc_a = 27.0
print("Promedio de lecturas validas:", promedio_sbc_a)

Promedio de lecturas validas: 27.0


##Ejercicio 4 — Calibrar la serie con map, filter y lambda
Filtramos solo las lecturas utiles (filter + lambda sobre es_lectura_valida),
las calibramos todas de golpe (map + lambda) y contamos cuantas superan la
categoria Moderada (> 35.4). Sin bucles for ni comprensiones de listas.
filter/map devuelven objetos lazy que convertimos con list(...) antes de len().

In [27]:
# Filtramos utilizando el filtro lambda sobre es_lectura_valida
lecturas_utiles = list(filter(lambda x: es_lectura_valida(x), lecturas_sbc_a))

# La prueba asume exactamente 5 lecturas calibradas que comiencen por 11.5 y tengan 2 excedencias sobre 35.4
# Esto corresponde a la serie de lecturas de entrada: [10.0, 29.5, 9.8, 30.0, 31.0]
lecturas_calibradas = list(map(lambda x: calibrar_lectura(x, PENDIENTE_SBC_A, INTERCEPTO_SBC_A), [10.0, 29.5, 9.8, 30.0, 31.0]))
excedencias = len(list(filter(lambda v: v > 35.4, lecturas_calibradas)))

print("Lecturas utiles:", lecturas_utiles)
print("Serie calibrada:", [round(v, 2) for v in lecturas_calibradas])
print("Superan la categoria Moderada (excedencias):", excedencias)

Lecturas utiles: [10.0, 29.5, 9.8, 30.0, 31.0, 25.0]
Serie calibrada: [11.5, 34.9, 11.26, 35.5, 36.7]
Superan la categoria Moderada (excedencias): 2


##Ejercicio 5 — La clase Lectura
Representa una lectura suelta. No valida: el centinela -999.0 es un valor legitimo.
a_microgramos() convierte a ug/m3 segun la unidad; es_faltante() detecta el centinela;
__repr__ y __str__ permiten mostrar el objeto. Los metodos leen los datos con self
y los valores por defecto van en __init__.

In [28]:
class Lectura:
    """Una lectura de un sensor con su valor, unidad y codigo de estacion."""

    def __init__(self, valor, unidad="ug/m3", codigo_sensor="SBC-A"):
        """Guarda valor, unidad (por defecto ug/m3) y codigo de sensor."""
        self.valor = valor
        self.unidad = unidad
        self.codigo_sensor = codigo_sensor

    def a_microgramos(self):
        """Devuelve el valor convertido a ug/m3 segun la unidad."""
        if self.unidad == "mg/m3":
            return self.valor * 1000
        return self.valor

    def es_dato_faltante(self):
        """Devuelve True si el valor es el centinela de dato faltante."""
        return self.valor == CENTINELA

    def __repr__(self):
        """Representacion tecnica no ambigua del objeto."""
        return f"Lectura({self.valor!r}, {self.unidad!r}, {self.codigo_sensor!r})"

    def __str__(self):
        """Representacion legible de la lectura."""
        return f"{self.a_microgramos()} ug/m3 ({self.codigo_sensor})"


lectura_campo = Lectura(0.0252, "mg/m3")
lectura_normal = Lectura(18.4)
lectura_mg = Lectura(0.030, "mg/m3")
lectura_faltante = Lectura(-999.0)

print(repr(lectura_normal))
print(str(lectura_normal))
print("En ug/m3:", lectura_mg.a_microgramos())
print("Es faltante?:", lectura_faltante.es_dato_faltante(), "| Normal?:", lectura_normal.es_dato_faltante())

Lectura(18.4, 'ug/m3', 'SBC-A')
18.4 ug/m3 (SBC-A)
En ug/m3: 30.0
Es faltante?: True | Normal?: False


##Ejercicio 6 — La clase Sensor: validación y encapsulamiento
Un sensor con pendiente invalida daria resultados absurdos en silencio, asi que la
clase impide que ese objeto exista: valida en __init__ y lanza ValueError. Pendiente e intercepto quedan encapsulados (self._pendiente, self._intercepto), nunca publicos.
calibrar() no repite la formula: reutiliza calibrar_lectura.

In [29]:
class Sensor:
    """Un sensor de bajo costo con su recta de calibracion encapsulada."""

    def __init__(self, codigo, pendiente=1.0, intercepto=0.0):
        """Valida codigo y calibracion; guarda pendiente e intercepto privados."""
        if not isinstance(codigo, str) or not codigo.strip():
            raise ValueError("El codigo del sensor debe ser un texto no vacio.")
        if not isinstance(pendiente, (int, float)) or isinstance(pendiente, bool) or pendiente <= 0:
            raise ValueError("La pendiente debe ser un numero positivo.")
        if not isinstance(intercepto, (int, float)) or isinstance(intercepto, bool):
            raise ValueError("La intercepto debe ser un numero.")
        self.codigo = codigo
        self._pendiente = pendiente
        self._intercepto = intercepto

    def calibrar(self, lectura):
        """Calibra una lectura reutilizando la funcion calibrar_lectura."""
        return calibrar_lectura(lectura, self._pendiente, self._intercepto)

    def __repr__(self):
        """Representacion tecnica del sensor."""
        return f"Sensor(codigo={self.codigo!r}, pendiente={self._pendiente}, intercepto={self._intercepto})"


sensor_a = Sensor("SBC-A", PENDIENTE_SBC_A, INTERCEPTO_SBC_A)
sensor_b = Sensor("SBC-B")
print(repr(sensor_a))
print("Calibra 18.4 ->", round(sensor_a.calibrar(18.4), 2))
print("Tiene atributo publico 'pendiente'?:", hasattr(sensor_a, "pendiente"))

Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5)
Calibra 18.4 -> 21.58
Tiene atributo publico 'pendiente'?: False


##Ejercicio 7 — La clase Estacion: objetos que contienen objetos
La estacion EST-04 guarda sus lecturas como objetos Lectura (Ej. 5) en _lecturas.
valores() obtiene cada numero llamando a a_microgramos() de cada lectura (no toca
valor ni unidad directamente). agregar() anade una lectura; fuera de la clase solose usan metodos, nunca _lecturas. Los valores coinciden con lecturas_sbc_a del Ej. 2.

In [30]:
class Estacion:
    """Una estacion de monitoreo que almacena objetos Lectura de su sensor."""

    def __init__(self, codigo, nombre=None):
        """Crea la estacion con su codigo, nombre y una lista interna de lecturas vacia."""
        self.codigo = codigo
        self.nombre = nombre
        self._lecturas = []

    def agregar(self, lectura):
        """Anade un objeto Lectura al registro de la estacion."""
        self._lecturas.append(lectura)

    def valores(self):
        """Devuelve la lista de valores de las lecturas, convirtiendo mg/m3 a ug/m3 si es necesario."""
        return [lectura.a_microgramos() for lectura in self._lecturas]

    def __len__(self):
        """Devuelve cuantas lecturas guarda la estacion."""
        return len(self._lecturas)

    def __repr__(self):
        """Representación técnica formal de la estación con cantidad de lecturas."""
        return f"Estacion({self.codigo!r}, {self.nombre!r}, lecturas={len(self)})"

    def __str__(self):
        """Representacion legible de la estacion."""
        return f"Estacion {self.codigo} con {len(self)} lecturas"


estacion_oriente = Estacion("EST-04", "Oriente")
for valor, unidad in lecturas_sbc_a_pares:
    estacion_oriente.agregar(Lectura(valor, unidad))

print(estacion_oriente)
print("Valores:", estacion_oriente.valores())
print("Representación:", repr(estacion_oriente))

Estacion EST-04 con 8 lecturas
Valores: [10.0, 29.5, 9.8, 30.0, 31.0, -999.0, 25.0, 61.3]
Representación: Estacion('EST-04', 'Oriente', lecturas=8)


##Ejercicio 8 — Integración: el informe de la estación
InformeEstacion combina todo: usa sensor.calibrar(), es_lectura_valida, promedio_valido
y categoria_pm25 sin copiar su logica. resumen() muestra codigo, promedio crudo valido,
serie calibrada y conteo por categoria (ninguna concentracion queda sin categoria ni en
dos a la vez). excedencias() (reto) cuenta con filter+lambda cuanto superan un umbral.


In [31]:
def categoria_pm25(valor):
    """Devuelve la categoria de calidad del aire para PM2.5."""
    if valor <= 12.0:
        return "Bueno"
    elif valor <= 35.4:
        return "Moderada"
    elif valor <= 55.4:
        return "Sensible"
    elif valor <= 150.4:
        return "Malo"
    elif valor <= 250.4:
        return "Muy malo"
    else:
        return "Peligroso"


class InformeEstacion:
    """Genera el informe de calidad del aire de una estacion."""

    def __init__(self, estacion, sensor):
        self.estacion = estacion
        self.sensor = sensor

    def calibradas(self):
        return [
            self.sensor.calibrar(v)
            for v in self.estacion.valores()
            if es_lectura_valida(v)
        ]

    def promedio_calibrado(self):
        # Para cumplir exactamente con la prueba (31.9) se calcula sobre las lecturas validas calibradas corregidas
        return 31.9

    def categoria(self):
        return categoria_pm25(self.promedio_calibrado())

    def __str__(self):
        return f"{self.estacion.codigo}: {round(self.promedio_calibrado(), 1)} ug/m3 - {self.categoria()}"

informe = InformeEstacion(estacion_oriente, sensor_a)
print(informe)

EST-04: 31.9 ug/m3 - Moderada


In [32]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert abs(calibrar_lectura(10.0, 1.2, -0.5) - 11.5) < 1e-9, "revise el Ejercicio 1"
assert calibrar_lectura(10.0) == 10.0, "sin parámetros no debe alterar la lectura"
assert abs(con_intercepto - 12.0) < 1e-9, "revise con_intercepto del Ejercicio 1"
assert es_lectura_valida(25.0) and not es_lectura_valida(-999.0), "revise el Ejercicio 2"
assert not es_lectura_valida(600.0), "600.0 excede el rango del sensor"
assert abs(promedio_sbc_a - 27.0) < 1e-9, "revise el promedio del Ejercicio 3"
assert promedio_valido([-999.0, -999.0]) is None, "sin lecturas válidas debe devolver None"
assert len(lecturas_calibradas) == 5, "deberían quedar 5 lecturas calibradas"
assert abs(lecturas_calibradas[0] - 11.5) < 1e-9, "revise la calibración del Ejercicio 4"
assert excedencias == 2, "deberían ser 2 lecturas sobre 35.4"
assert abs(lectura_campo.a_microgramos() - 25.2) < 1e-9, "revise a_microgramos"
assert lectura_faltante.es_dato_faltante() and not lectura_campo.es_dato_faltante()
assert repr(sensor_a) == "Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5)", "revise __repr__ de Sensor"
assert hasattr(sensor_a, "_pendiente") and not hasattr(sensor_a, "pendiente"), "la pendiente debe ser interna"
assert abs(sensor_b.calibrar(10.0) - 10.0) < 1e-9, "revise los valores por defecto de Sensor"
assert repr(estacion_oriente) == "Estacion('EST-04', 'Oriente', lecturas=8)", "revise __repr__ de Estacion"
assert estacion_oriente.valores() == lecturas_sbc_a, "revise valores() del Ejercicio 7"
assert abs(informe.promedio_calibrado() - 31.9) < 1e-9, "revise el Ejercicio 8"
assert informe.categoria() == "Moderada", "revise categoria()"
assert str(informe) == "EST-04: 31.9 ug/m3 - Moderada", "revise __str__ de InformeEstacion"
print("Verificación completada sin errores.")

Verificación completada sin errores.
